# QuranicTools — Full Functionality Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/language-ml/quranic-nlp/blob/main/notebooks/quranic_nlp_demo.ipynb)

This notebook demonstrates all available features of the `quranic-nlp` library.

**Contents:**
1. [Installation & Data Download](#1-installation--data-download)
2. [Pipeline Setup](#2-pipeline-setup)
3. [Input Formats](#3-input-formats)
4. [Verse-Level Information](#4-verse-level-information)
5. [Token-Level Analysis](#5-token-level-analysis)
6. [Dependency Parse Visualization](#6-dependency-parse-visualization)
7. [JSON Output](#7-json-output)
8. [Similar Verses](#8-similar-verses)
9. [Translations](#9-translations)
10. [Multiple Matches](#10-multiple-matches)
11. [Hadiths](#11-hadiths)
12. [Surah-Level Graph Analysis](#12-surah-level-graph-analysis)

## 1. Installation & Data Download

In [ ]:
!pip install quranic-nlp==1.3.8

In [ ]:
from quranic_nlp.data_requirements import download_data
download_data()

## 2. Pipeline Setup

Available pipeline components:
- `dep` — Dependency parsing
- `pos` — Part-of-speech tagging
- `root` — Root extraction
- `lem` — Lemmatization

In [ ]:
from quranic_nlp import language, utils, constant

# Full pipeline with Persian (Farsi) translation, translator index 1
# hadiths=False (default) — skips hadith fetching for fast verse processing
translation_lang = 'fa#1'
pips = 'dep,pos,root,lem'

nlp = language.Pipeline(pips, translation_lang, hadiths=False)
print('Pipeline ready.')

## 3. Input Formats

The pipeline accepts four input formats:

| Format | Example | Returns |
|--------|---------|---------|
| `surah_number#ayah_number` | `nlp('1#1')` | single `Doc` |
| `surah_name#ayah_number` | `nlp('حمد#1')` | single `Doc` |
| surah name or index + `surah=True` | `nlp('فاتحه', surah=True)` | `SurahDoc` (all verses) |
| free Arabic text | `nlp('رب العالمین')` | `list[Doc]` (all matches) |

In [ ]:
# Format 1: surah_number#ayah_number  →  single Doc
doc = nlp('1#1')
print('surah:', doc._.surah, '| ayah:', doc._.ayah, '| text:', doc._.text)

In [ ]:
# Format 2: surah_name#ayah_number  →  single Doc  (requires internet)
doc = nlp('حمد#1')
print('surah:', doc._.surah, '| ayah:', doc._.ayah, '| text:', doc._.text)

In [ ]:
# Format 3: surah name or integer index with surah=True  →  SurahDoc (all verses)
surah_fatiha = nlp('فاتحه', surah=True)   # same as nlp(1, surah=True) or nlp('1', surah=True)
print(repr(surah_fatiha))
for v in surah_fatiha:
    print(f'  Ayah {v._.ayah}: {v._.text}')

In [ ]:
# Format 4: free Arabic text  →  list[Doc] of all matching verses  (requires internet)
results = nlp('رب العالمین')
print(f'Found {len(results)} matching verse(s):')
for v in results[:5]:
    print(f'  {v._.surah} {v._.ayah}: {v._.text}')

## 4. Verse-Level Information

Each processed verse (`Doc`) carries surah/ayah metadata as spaCy custom extensions.

In [ ]:
# Surah Al-Fatiha, Verse 1  —  used throughout this notebook
doc1 = nlp('1#1')

print('Text (with diacritics)    :', doc1._.text)
print('Text (without diacritics) :', doc1._.simple_text)
print('Surah name                :', doc1._.surah)
print('Surah index               :', doc1._.soure_index)
print('Ayah number               :', doc1._.ayah)
print('Revelation order          :', doc1._.revelation_order)
print('Translation               :', doc1._.translations)

In [ ]:
# Another example — Surah Aal-i-Imran, Verse 200
doc2 = nlp('3#200')

print('Text (with diacritics)    :', doc2._.text)
print('Text (without diacritics) :', doc2._.simple_text)
print('Surah name                :', doc2._.surah)
print('Surah index               :', doc2._.soure_index)
print('Ayah number               :', doc2._.ayah)
print('Revelation order          :', doc2._.revelation_order)
print('Translation               :', doc2._.translations)

## 5. Token-Level Analysis

The pipeline works on **morphologically segmented tokens** — each word is split into its stems and affixes for linguistic analysis. `str(token)` gives the segmented form; use `doc._.text` for the original verse text.

| Attribute | Description |
|-----------|-------------|
| `doc._.text` | Original verse text **with** diacritics |
| `strip_diacritics(doc._.text)` | Original verse text **without** diacritics |
| `str(token)` | Morphological segment (stem/affix) with diacritics |
| `strip_diacritics(token)` | Morphological segment without diacritics |
| `token.lemma_` | Dictionary lemma |
| `token._.root` | Trilateral Arabic root |
| `token.pos_` | Universal POS tag |
| `constant.POS_UNI_FA[token.pos_]` | POS label in Arabic |
| `token.dep_` | Syntactic dependency relation |
| `token._.dep_arc` | Arc direction (`LTR` or `RTL`) |
| `token.head` | Syntactic head token |

In [ ]:
import re

def strip_diacritics(text):
    """Remove Arabic diacritics (ḥarakāt, sukūn, shadda, etc.)."""
    return re.sub(r'[\u0610-\u061A\u064B-\u065F\u0670]', '', str(text))

# Single token deep-dive — third token of Surah Al-Fatiha, Verse 1
token = doc1[2]

print('═' * 45)
print(f'  Token #{token.i + 1} of {len(doc1)}')
print('═' * 45)
print(f'  With diacritics     : {str(token)}')
print(f'  Without diacritics  : {strip_diacritics(token)}')
print(f'  Lemma               : {token.lemma_}')
print(f'  Root                : {token._.root}')
print(f'  POS (Universal)     : {token.pos_}')
print(f'  POS (Arabic)        : {constant.POS_UNI_FA.get(token.pos_, "—")}')
print(f'  Dependency          : {token.dep_}')
print(f'  Arc direction       : {token._.dep_arc}')
print(f'  Head                : {token.head}')

In [ ]:
# All tokens in Surah Al-Fatiha, Verse 1 — full comparison table
col = {'w_diac': 22, 'wo_diac': 18, 'lemma': 18, 'root': 10, 'pos_u': 8, 'pos_ar': 12, 'dep': 14, 'arc': 6, 'head': 18}

header = (f"{'With diacritics':<{col['w_diac']}}  "
          f"{'No diacritics':<{col['wo_diac']}}  "
          f"{'Lemma':<{col['lemma']}}  "
          f"{'Root':<{col['root']}}  "
          f"{'POS':<{col['pos_u']}}  "
          f"{'POS (Arabic)':<{col['pos_ar']}}  "
          f"{'Dependency':<{col['dep']}}  "
          f"{'Arc':<{col['arc']}}  "
          f"{'Head':<{col['head']}}")
print(header)
print('─' * len(header))

for t in doc1:
    pos_ar = constant.POS_UNI_FA.get(t.pos_, '—')
    print(f"{str(t):<{col['w_diac']}}  "
          f"{strip_diacritics(t):<{col['wo_diac']}}  "
          f"{t.lemma_:<{col['lemma']}}  "
          f"{str(t._.root):<{col['root']}}  "
          f"{t.pos_:<{col['pos_u']}}  "
          f"{pos_ar:<{col['pos_ar']}}  "
          f"{t.dep_:<{col['dep']}}  "
          f"{str(t._.dep_arc):<{col['arc']}}  "
          f"{str(t.head):<{col['head']}}")

## 6. Dependency Parse Visualization

Render the syntactic dependency tree using spaCy's `displacy`. Arrows connect each token to its syntactic head, labelled with the dependency relation.

In [ ]:
from spacy import displacy

options = {
    'compact': True,
    'bg': '#09a3d5',
    'color': 'white',
    'font': 'Arial'
}

# Surah Al-Fatiha, Verse 1 — dependency parse tree
displacy.render(doc1, style='dep', options=options, jupyter=True)

## 7. JSON Output

`language.to_json()` serializes all token-level information to a list of dicts — convenient for downstream processing or export.

In [ ]:
import json

result = language.to_json(pips, doc1)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 8. Similar Verses

`doc._.sim_ayahs` returns a ranked list of `(ref, score)` tuples — verses from across the Quran that are semantically similar to the current verse.

In [ ]:
# Top 10 similar verses to Surah Al-Fatiha, Verse 1
print(f'Verse: {doc1._.text}')
print()
print(f'{"Ref":<12}  {"Score":>8}  Text')
print('─' * 70)
for ref, score in doc1._.sim_ayahs[:10]:
    # Load the similar verse text
    soure, ayeh = (int(x) for x in ref.split('#'))
    from quranic_nlp import utils
    text = utils.get_text(soure, ayeh) or ''
    print(f'{ref:<12}  {score:>8.4f}  {text[:45]}')

## 9. Translations

The library supports 43 languages. Use `utils.print_all_translations()` to list all available translators.

- Passing `'lang#index'` (e.g. `'fa#1'`) returns a **single string** from that translator.
- Passing `'lang'` only (e.g. `'fa'`) returns a **dict** keyed by translator name with all translations for that language.

In [ ]:
utils.print_all_translations()

In [ ]:
# Single translator — returns a string
nlp_en = language.Pipeline(pips, 'en#16')   # English (Yusuf Ali)
doc_en = nlp_en('1#1')
print('English (Yusuf Ali):', doc_en._.translations)

In [ ]:
# All translators for a language — returns a dict {translator_name: text}
nlp_fa = language.Pipeline(pips, 'fa')
doc_fa = nlp_fa('1#1')
print('All Persian translations of Surah Al-Fatiha, Verse 1:\n')
for name, text in doc_fa._.translations.items():
    print(f'  {name}:')
    print(f'    {text}')

## 10. Multiple Matches

Free-text search may match many verses across the Quran. Use `language.search_all(nlp, text)` to retrieve all of them, or pass `max_results` to limit the count.

In [ ]:
docs = language.search_all(nlp, 'رب العالمین', max_results=5)

print(f'Found {len(docs)} matching verse(s):\n')
for d in docs:
    print(f'  {d._.surah} ({d._.ayah}): {d._.text}')

## 13. Token Pattern Queries

`quranic_nlp.query` provides spaCy-style pattern matching **within verses**.
Patterns filter on `ROOT`, `LEMMA`, `POS`, `DEP`, `TEXT`, with `SKIP` for proximity and `OP` for quantifiers.


In [ ]:
from quranic_nlp import query

nlp = language.Pipeline('pos,root,lem,dep')
matcher = query.VerseMatcher(nlp)

# All verses in Al-Fatiha that have a NOUN with root رحم
matcher.add('MERCY_NOUN', [[{'ROOT': 'رحم', 'POS': 'NOUN'}]])

# Root رحم within 5 tokens of lemma الله
matcher.add('MERCY_NEAR_ALLAH', [[
    {'ROOT': 'رحم'},
    {'LEMMA': 'ٱللَّه', 'SKIP': 5},
]])

for doc, matches in matcher.search(surah=1):
    for key, start, end in matches:
        print(key, f"{doc._.surah}:{doc._.ayah}", doc[start:end])


In [ ]:
# Find all verses in Al-Baqarah (surah 2) where root رحم appears near root علم (within 5 tokens)
results = query.find_near(
    nlp, {'ROOT': 'رحم'}, {'ROOT': 'علم'}, max_dist=5, surah=2
)
for doc, s1, e1, s2, e2 in results[:5]:
    print(f"{doc._.surah}:{doc._.ayah}  {doc[s1:e1]} … {doc[s2:e2]}")


## 14. Cross-Verse Corpus Search

`quranic_nlp.corpus.CorpusIndex` treats the **entire Quran as one flat sequence** of ~128 K tokens.  
Patterns can span verse and surah boundaries. Inverted numpy indexes make each step O(log N).

**TAG notation** (Quranic Treebank): `N` noun · `V` verb · `P` preposition · `PN` proper noun · `PRON` pronoun · `CONJ` conjunction · `DET` determiner · `ADJ` adjective · `NEG` negation


In [ ]:
from quranic_nlp.corpus import CorpusIndex

# Build once (~1-2 s) and cache to disk; subsequent loads take ~0.04 s
idx = CorpusIndex.build(save=True, verbose=True)
print(idx)
# → CorpusIndex(N=128,219)


In [ ]:
# All occurrences of root رحم
matches = idx.find_root('رحم', max_results=5)
for m in matches:
    print(m)
# CorpusMatch(key='ROOT:رحم', refs=[1:1], text='رَّحْمَٰنِ')
# CorpusMatch(key='ROOT:رحم', refs=[1:1], text='رَّحِيمِ')
# CorpusMatch(key='ROOT:رحم', refs=[1:3], text='رَّحْمَٰنِ')
# CorpusMatch(key='ROOT:رحم', refs=[1:3], text='رَّحِيمِ')
# CorpusMatch(key='ROOT:رحم', refs=[2:37], text='رَّحِيمُ')


In [ ]:
# Root رحم within 5 tokens of root علم  -- can cross verse boundaries
matches = idx.find_root_near_root('رحم', 'علم', max_dist=5, max_results=5)
for m in matches:
    print(m)
    for t in m.tokens:
        print(f"  {t.soure}:{t.ayeh} tok={t.tok_i}  {t.text!r:22}  root={t.root!r}  tag={t.tag}")


In [ ]:
# Noun صبر followed within 3 tokens by a verb (cross-verse allowed)
matches = idx.search([
    {'TAG': 'N', 'ROOT': 'صبر'},
    {'TAG': 'V', 'SKIP': 3},
], max_results=5)
for m in matches:
    print(m)

# Optional DET between root علم and a noun
matches = idx.search([
    {'ROOT': 'علم'},
    {'TAG': 'DET', 'OP': '?'},
    {'TAG': 'N'},
], max_results=5)
for m in matches:
    print(m)

# Any-of roots (IN syntax)
matches = idx.search([{'ROOT': {'IN': ['رحم', 'غفر', 'توب']}}], max_results=5)
for m in matches:
    print(m)


In [ ]:
# Find the famous cross-verse pair in Surah Al-Rahman (55:1-2)
matches = idx.search([
    {'ROOT': 'رحم'},
    {'ROOT': 'علم', 'SKIP': 1},
])
for m in matches:
    if len(m.refs) > 1:
        print("Cross-verse match:", m)
        print("  refs:", m.refs)
        for t in m.tokens:
            print(f"  soure={t.soure} ayeh={t.ayeh}  {t.text!r:22}  simple={t.simple!r}  root={t.root!r}")
# → Cross-verse match: CorpusMatch(key='match', refs=[55:1, 55:2], text='رَّحْمَٰنُ عَلَّمَ')


## 11. Hadiths

Hadith fetching is **disabled by default** () to avoid slow HTTP requests during surah-level processing.
Enable it explicitly with  when creating the pipeline.

 is  unless the pipeline was created with .

In [ ]:
# Create a pipeline with hadith fetching enabled
nlp_hadiths = language.Pipeline(pips, translation_lang, hadiths=True)
doc_h = nlp_hadiths('1#1')

hadiths = doc_h._.hadiths
if hadiths:
    print(f'Found {len(hadiths)} hadith(s):')
    for i, h in enumerate(hadiths[:2], 1):
        print(f'--- Hadith {i} ---')
        print(h)
        print()
else:
    print('No hadiths found or API unavailable.')

## 12. Surah-Level Graph Analysis

Calling `nlp('فاتحه')` (or `nlp(1)`) returns a **`SurahDoc`** — an object containing all verse docs for the surah along with graph-based similarity analysis tools.

The graph is built from pairwise cosine similarity between verses. Two representation methods are available:
- **`'tfidf'`** — TF-IDF over concatenated surface form + lemma + root tokens per verse
- **`'embedding'`** — cosine similarity of sentence embeddings (any model with `.encode()`)

In [ ]:
# Get all verses of a surah as a SurahDoc using surah=True
surah = nlp('فاتحه', surah=True)   # same as nlp(1, surah=True) or nlp('1', surah=True)

print(repr(surah))
print(f'Surah: {surah.surah}')
print(f'Number of verses: {len(surah)}')
print()

# Iterate over verse docs
for doc in surah:
    print(f'  Ayah {doc._.ayah}: {doc._.text}')

### Build a Verse-Similarity Graph

`build_graph()` computes pairwise cosine similarity between verses using TF-IDF over concatenated surface form + lemma + root tokens. Each node is a verse; each edge weight is the cosine similarity.

In [ ]:
import networkx as nx

# Build TF-IDF similarity graph (threshold=0.0 keeps all edges)
G = surah.build_graph(rep='tfidf', threshold=0.0)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print()

# Show edge weights between verses
print(f'{"Ayah i":>8}  {"Ayah j":>8}  {"Similarity":>10}')
print('-' * 32)
for u, v, data in sorted(G.edges(data=True), key=lambda x: -x[2]['weight']):
    print(f'{u+1:>8}  {v+1:>8}  {data["weight"]:>10.4f}')

### Central Verse

Find the most central verse using different graph centrality measures.

In [ ]:
# Compare all centrality methods
methods = ['pagerank', 'degree', 'betweenness', 'eigenvector', 'mst']

print(f'{"Method":<14}  {"Central Ayah":>12}  {"Text"}')
print('-' * 70)
for method in methods:
    doc, scores = surah.central_verse(method=method)
    print(f'{method:<14}  Ayah {doc._.ayah:>2}        {doc._.text[:50]}')

print()

# Detailed scores for PageRank
doc, scores = surah.central_verse(method='pagerank')
print('PageRank scores per verse:')
for idx, score in sorted(scores.items(), key=lambda x: -x[1]):
    marker = ' ← most central' if idx == list(scores.keys())[list(scores.values()).index(max(scores.values()))] else ''
    print(f'  Ayah {idx+1}: {score:.4f}{marker}')

print()
print(f'Most central verse (PageRank):')
print(f'  Ayah {doc._.ayah}: {doc._.text}')
if doc._.translations:
    print(f'  Translation: {doc._.translations}')

### Maximum Spanning Tree

The MST keeps only the strongest similarity connections — useful for visualizing the backbone structure of a surah.

In [ ]:
T = surah.mst()

print(f'MST: {T.number_of_nodes()} nodes, {T.number_of_edges()} edges')
print()
print('MST edges (strongest similarity links):')
for u, v, data in sorted(T.edges(data=True), key=lambda x: x[2].get('weight', 0)):
    # MST stores negated weights internally; recover original similarity
    sim = -data.get('weight', 0)
    print(f'  Ayah {u+1} ↔ Ayah {v+1}  similarity={sim:.4f}')

### Visualize the Graph

Plot the full similarity graph and MST with properly rendered Arabic text labels.

In [ ]:
!pip install arabic-reshaper python-bidi -q

import arabic_reshaper
from bidi.algorithm import get_display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

def ar(text):
    """Reshape and apply BiDi algorithm for correct Arabic rendering in matplotlib."""
    return get_display(arabic_reshaper.reshape(str(text)))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Node labels: Ayah number prominently + first few chars of Arabic text
def make_label(i):
    text = surah.docs[i]._.simple_text or surah.docs[i]._.text or ''
    short = text[:10] + ('…' if len(text) > 10 else '')
    return f"آیه {i+1}\n{ar(short)}"

labels = {i: make_label(i) for i in range(len(surah))}

# ── Full similarity graph ──────────────────────────────────────────────────
ax1 = axes[0]
G = surah.graph
pos = nx.spring_layout(G, seed=42, weight='weight')

weights = [G[u][v]['weight'] for u, v in G.edges()] or [0]
nx.draw_networkx_nodes(G, pos, ax=ax1, node_size=1600, node_color='#3a7ecf', alpha=0.9)
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax1, font_size=8, font_color='white')
nx.draw_networkx_edges(G, pos, ax=ax1, width=[w * 5 for w in weights],
                       edge_color=weights, edge_cmap=plt.cm.Blues,
                       edge_vmin=0, edge_vmax=1, alpha=0.75)
edge_labels = {(u, v): f'{G[u][v]["weight"]:.2f}' for u, v in G.edges()}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax1, font_size=7)
ax1.set_title(ar(f'نمودار شباهت — {surah.surah}'), fontsize=13, pad=12)
ax1.axis('off')

# ── Maximum Spanning Tree ──────────────────────────────────────────────────
ax2 = axes[1]
T = surah.mst()
pos_t = nx.spring_layout(T, seed=42)
mst_weights = [-T[u][v].get('weight', 0) for u, v in T.edges()] or [0]
nx.draw_networkx_nodes(T, pos_t, ax=ax2, node_size=1600, node_color='#c95d2a', alpha=0.9)
nx.draw_networkx_labels(T, pos_t, labels=labels, ax=ax2, font_size=8, font_color='white')
nx.draw_networkx_edges(T, pos_t, ax=ax2, width=[w * 5 for w in mst_weights],
                       edge_color=mst_weights, edge_cmap=plt.cm.Oranges,
                       edge_vmin=0, edge_vmax=1, alpha=0.85)
mst_edge_labels = {(u, v): f'{-T[u][v].get("weight", 0):.2f}' for u, v in T.edges()}
nx.draw_networkx_edge_labels(T, pos_t, edge_labels=mst_edge_labels, ax=ax2, font_size=7)
ax2.set_title(ar(f'درخت پوشای بیشینه — {surah.surah}'), fontsize=13, pad=12)
ax2.axis('off')

plt.tight_layout()
plt.show()

### Using a Sentence Embedding Model (optional)

You can replace TF-IDF with a sentence embedding model for deeper semantic similarity. Any model with a `.encode(list[str]) -> np.ndarray` method works (e.g. `sentence-transformers`).

```python
# pip install sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('CAMeL-Lab/bert-base-arabic-camelbert-ca')
G = surah.build_graph(rep='embedding', model=model, threshold=0.5)
doc, scores = surah.central_verse(method='pagerank')
print(doc._.ayah, doc._.text)
```

You can also call the lower-level `graph` module directly with any list of docs:

```python
from quranic_nlp import language, graph

nlp2 = language.Pipeline('pos,root,lem')
docs = language.surah_docs(nlp2, 'بقره')   # Surah Al-Baqarah (index 2)
G = graph.build_graph(docs, rep='tfidf', threshold=0.1)
T = graph.mst(G)
doc, scores = graph.central_verse(G, docs, method='eigenvector')
print(doc._.ayah, doc._.text)
```